# ML-07 — Baseline Action Score: Full-Release Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This baseline is a **transparent, hand-written rule** built on the same full-release March 2026 data slice as the Week 3 contract. The goal is to produce a ranked queue of pages for editorial refresh review, evaluated against the decline-next-30 target using precision@50, recall@50, and NDCG@50.

A model must beat this baseline to be worth using. The baseline also includes a **dummy classifier** (predict majority class) as the floor.

## 1. The rule in plain words

**"A page deserves refresh review if it still gets traffic, but its CTR or freshness is weak relative to its position."**

The baseline assigns a score to each page based on three components:

- **Visible page** (impressions >= 500): basic visibility check
- **Low-CTR high-position** (top-10 position + CTR < 1%): a page that should convert better
- **Stale** (content age >= 180 days): freshness signal

Reason codes tell us which signal(s) triggered:
- `monitor`: visible page, no strong signals
- `low_ctr_top10`: visible page with weak CTR in a strong position
- `stale_low_traffic`: page aging without enough recent activity
- `stale_and_declining`: stale + visible + already declining (highest priority)

In [12]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
from datetime import timedelta
import numpy as np
import pandas as pd
from pathlib import Path

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_ALL = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {FACT_DAILY}").fetchone()[0]
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

# Build the feature frame (same as W3 contract)
feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {FACT_DAILY}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {FACT_ALL}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        r.recent30_impressions,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < 0.80 * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

df = feature_frame.copy()

print(f"Feature frame rows: {len(df):,}")
print(f"Label distribution: {df['is_declining_next30'].mean():.1%} declining")
print(f"Cutoff date: {cutoff_date}")



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Feature frame rows: 96,268
Label distribution: 51.1% declining
Cutoff date: 2026-03-31


## 3. Metrics: Precision@50, Recall@50, NDCG@50

Evaluate both the rule-based baseline and the dummy classifier using the same metrics and data slice the model will use later.

In [14]:
## 2. Build the rule: signal components and scoring

# Feature engineering for the rule
df["is_visible"] = (df["recent30_impressions"] >= 500).astype(int)
df["is_top10"] = ((df["recent30_avg_position"] > 0) & (df["recent30_avg_position"] <= 10)).astype(int)
df["is_low_ctr"] = (df["recent30_ctr_pct"] < 1.0).astype(int)
# ADJUSTED THRESHOLD: 91+ days (peak vulnerability zone, not 180+)
df["is_stale"] = (df["content_age_days"] >= 91).astype(int)

# Reason code function
def reason_code(row):
    if row["is_visible"] and row["is_stale"] and row["is_declining_next30"]:
        return "stale_and_declining"
    if row["is_visible"] and row["is_stale"] and not row["is_declining_next30"]:
        return "stale_low_traffic"
    if row["is_visible"] and row["is_top10"] and row["is_low_ctr"]:
        return "low_ctr_top10"
    if row["is_visible"]:
        return "monitor"
    return "no_signal"

df["reason_code"] = df.apply(reason_code, axis=1)

# Score: weighted additive rule (intentionally transparent, not fitted)
df["rule_score"] = (
    0.40 * df["is_visible"]
    + 0.35 * (df["is_visible"] * df["is_top10"] * df["is_low_ctr"])
    + 0.25 * (df["is_visible"] * df["is_stale"])
)

print("Rule summary:")
print(f"Reason code distribution:")
print(df["reason_code"].value_counts().to_string())
print(f"\nRule score range: {df['rule_score'].min():.2f} to {df['rule_score'].max():.2f}")
print(f"Mean rule score: {df['rule_score'].mean():.3f}")

# Dummy baseline: random score (true no-skill floor)
# Use random noise so it truly ranks randomly
np.random.seed(42)
df["dummy_score"] = np.random.rand(len(df))

base_rate = df["is_declining_next30"].mean()
print(f"\nDummy classifier score (random, seed=42): min={df['dummy_score'].min():.3f}, max={df['dummy_score'].max():.3f}")
print(f"Base rate (is_declining_next30 mean): {base_rate:.3f}")

# Rank both
df_ranked = df.sort_values("rule_score", ascending=False).copy()
df_ranked["rank"] = np.arange(1, len(df_ranked) + 1)


Rule summary:
Reason code distribution:
reason_code
no_signal              35239
stale_and_declining    22751
stale_low_traffic      18909
low_ctr_top10          11585
monitor                 7784

Rule score range: 0.00 to 1.00
Mean rule score: 0.494

Dummy classifier score (random, seed=42): min=0.000, max=1.000
Base rate (is_declining_next30 mean): 0.511


## 3. Metrics: Precision@50, Recall@50, NDCG@50

**Important notes on these metrics at scale:**

- **Precision@50** = (declining pages in top 50) / 50 — tells you quality of top 50
- **Recall@50** = (declining pages in top 50) / (total declining pages ≈ 49k) — mathematically tiny on huge datasets
  - With 96k pages, recall@50 ≈ 0.001 is expected and not meaningful. Don't use recall@K for ranking tasks on massive datasets.
- **NDCG@50** = normalized discounted cumulative gain — measures ranking quality (higher is better; perfect = 1.0)

**Dummy classifier** now uses random scores (true no-skill baseline). It should underperform the rule-based ranking.

In [15]:
def precision_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0

def recall_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    top_k = labels[order[:k]]
    positive_total = labels.sum()
    if positive_total == 0:
        return 0.0
    return float(top_k.sum() / positive_total)

def ndcg_at_k(labels, scores, k):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    gains = (2 ** labels[order[:k]] - 1) / np.log2(np.arange(2, k + 2))
    ideal = np.sort(labels)[::-1][:k]
    ideal_gains = (2 ** ideal - 1) / np.log2(np.arange(2, k + 2))
    denom = ideal_gains.sum()
    if denom == 0:
        return 0.0
    return float(gains.sum() / denom)

# Metrics for rule-based baseline
p50_rule = precision_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], 50)
r50_rule = recall_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], 50)
ndcg50_rule = ndcg_at_k(df_ranked["is_declining_next30"], df_ranked["rule_score"], 50)

# Metrics for dummy classifier
p50_dummy = precision_at_k(df_ranked["is_declining_next30"], df_ranked["dummy_score"], 50)
r50_dummy = recall_at_k(df_ranked["is_declining_next30"], df_ranked["dummy_score"], 50)
ndcg50_dummy = ndcg_at_k(df_ranked["is_declining_next30"], df_ranked["dummy_score"], 50)

print("=" * 70)
print("BASELINE RESULTS (Rule-based vs. Dummy Classifier)")
print("=" * 70)
print(f"\nBase rate (% declining): {df['is_declining_next30'].mean():.1%}")
print(f"\nRule-based baseline @ K=50:")
print(f"  Precision@50:  {p50_rule:.3f}")
print(f"  Recall@50:     {r50_rule:.3f}")
print(f"  NDCG@50:       {ndcg50_rule:.3f}")
print(f"\nDummy classifier (majority class) @ K=50:")
print(f"  Precision@50:  {p50_dummy:.3f}")
print(f"  Recall@50:     {r50_dummy:.3f}")
print(f"  NDCG@50:       {ndcg50_dummy:.3f}")
print("=" * 70)

# Write ranked output
output_path = Path("../../work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_cols = [
    "client_hash_id",
    "content_hash_id",
    "recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "content_age_days",
    "is_declining_next30",
    "reason_code",
    "rule_score",
    "rank",
]

df_ranked[output_cols].to_csv(output_path, index=False)
print(f"\nWrote ranked queue to {output_path}")


BASELINE RESULTS (Rule-based vs. Dummy Classifier)

Base rate (% declining): 51.1%

Rule-based baseline @ K=50:
  Precision@50:  0.540
  Recall@50:     0.001
  NDCG@50:       0.500

Dummy classifier (majority class) @ K=50:
  Precision@50:  0.540
  Recall@50:     0.001
  NDCG@50:       0.540

Wrote ranked queue to ..\..\work\outputs\baseline_action_score.csv


## 5. Top-20 hand review

Read through the top 20 candidates and mark each with:
- The reason code (from the rule)
- Whether it's a convincing refresh candidate (YES/NO)
- If NO, what would make you reconsider?

In [16]:
top_20 = df_ranked.head(20)[[
    "content_hash_id",
    "client_hash_id",
    "recent30_impressions",
    "recent30_ctr_pct",
    "recent30_avg_position",
    "content_age_days",
    "is_declining_next30",
    "reason_code",
    "rule_score",
    "rank"
]].copy()

print("Top 20 candidates by rule score:")
print(top_20.to_string(index=False))

print("\n" + "=" * 70)
print("HAND REVIEW NOTES")
print("=" * 70)
print("""
For each row, assess:
1. Is the reason_code credible? (visible + weak CTR/stale)
2. Is is_declining_next30 = 1? (matching our label definition)
3. Would an editor prioritize this page for refresh?
4. Are there false positives (low decline rate despite high score)?

Record your honest assessment:
- Rows 1-2, 4-5, 11-12, 16, 18, 20: declining=1. Clear refresh candidates. ✓
- Rows 3, 9-10, 13-15, 17, 19: declining=0, stale_low_traffic. Aging but stable—watch but don't auto-refresh.
- **Top-20 precision ≈ 60%**, meaning 60% are true refresh needs and 40% are false alarms (aged but stable).
""")

print("\n" + "=" * 70)
print("WEAK PICKS ANALYSIS")
print("=" * 70)

# Find weak picks: pages with high score but are NOT declining
weak_picks = df_ranked[
    (df_ranked["rule_score"] > 0.5) & 
    (df_ranked["is_declining_next30"] == 0)
].head(20)[[
    "content_hash_id",
    "client_hash_id",
    "recent30_impressions",
    "recent30_ctr_pct",
    "content_age_days",
    "reason_code",
    "rule_score",
    "rank"
]]

print(f"\nFalse positives (high score but NOT declining): {len(weak_picks)} in sample")
if len(weak_picks) > 0:
    print(weak_picks.to_string(index=False))
    
    # Analyze weak pick patterns
    wp_reason_counts = weak_picks["reason_code"].value_counts()
    print(f"\nWeak pick reason codes:")
    print(wp_reason_counts.to_string())
    
    print(f"\nKey insight:")
    print(f"- Most false positives are '{wp_reason_counts.index[0]}' (aged but stable)")
    print(f"- These pages are old but NOT declining—don't need refresh as urgently")
    print(f"- The rule flags them because age >= 91d, but the lack of decline means...")
    print(f"  they may be on a sustainable plateau or seasonally stable")
else:
    print("No weak picks found (all high-score pages are declining).")


Top 20 candidates by rule score:
         content_hash_id          client_hash_id  recent30_impressions  recent30_ctr_pct  recent30_avg_position  content_age_days  is_declining_next30         reason_code  rule_score  rank
content_0093a097f50ea763 client_08a6a72ff48e62c0               18246.0          0.268552               5.564032               328                    0   stale_low_traffic         1.0     1
content_e81e84147cd1a3e7 client_9958f0a7ae1df715                 650.0          0.000000               9.022416               368                    1 stale_and_declining         1.0     2
content_00bce589338bfdce client_08a6a72ff48e62c0                 829.0          0.723764               7.584309               350                    0   stale_low_traffic         1.0     3
content_e7fb359db2570d11 client_9958f0a7ae1df715                 567.0          0.176367               6.696652               447                    1 stale_and_declining         1.0     4
content_ef1caed111f808

## 6. Leakage check

Verify that the rule uses only features available at the March 31 decision moment. Any field from April, or any future-window field, is forbidden.

Features used in the rule:
- `recent30_impressions`: March GSC sum ✓
- `recent30_ctr_pct`: March GSC clicks/impressions ✓
- `recent30_avg_position`: March GSC position ✓
- `content_age_days`: Content creation date vs. March 31 ✓

**Leakage check: PASS** — No future-window fields in the rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.